# API Results Important Insights Analysis

This notebook mirrors the thesis-style local-model insights notebook for the API benchmark outputs in `02ScalingPhrase/apimodel/run_apis/results_final`.

It focuses on interpretation rather than only raw score tables: task-domain split, model-by-domain ranking, pairwise disagreement, category sensitivity, response cleanliness, and runtime/token behavior.

The notebook is dependency-light and uses only the Python standard library plus IPython display utilities.


In [ ]:
from __future__ import annotations

import csv
import html
import json
import statistics
from collections import Counter, defaultdict
from pathlib import Path
from IPython.display import HTML, Markdown, display


def find_results_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / '02ScalingPhrase' / 'apimodel' / 'run_apis' / 'results_final',
        Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd(),
    ]
    for candidate in candidates:
        if (candidate / 'deepseek-v4-flash.python.jsonl').exists() and (candidate / 'gemini-flash.python.jsonl').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not locate run_apis/results_final JSONL files')

RESULTS_DIR = find_results_dir()
ANALYSIS_DIR = RESULTS_DIR / 'analysis'

def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def load_rows() -> list[dict]:
    rows = []
    for path in sorted(RESULTS_DIR.glob('*.jsonl')):
        stem = path.stem
        if stem.endswith('.python'):
            model = stem[:-len('.python')]
            kind = 'python'
        elif stem.endswith('.math'):
            model = stem[:-len('.math')]
            kind = 'math'
        else:
            continue
        for row in read_jsonl(path):
            row = dict(row)
            row['model_slug'] = model
            row['kind'] = kind
            row['row_id'] = str(row.get('row_key') or f"{row.get('task_id')}::{row.get('category_id')}")
            row['category_id'] = str(row.get('category_id'))
            rows.append(row)
    return rows

rows = load_rows()

def fnum(value, default=0.0):
    try:
        return float(value)
    except Exception:
        return default

def pct(value):
    return f'{100 * fnum(value):.1f}%'

def pct_from_counts(passed, total):
    return f'{100 * passed / total:.1f}%' if total else '0.0%'

def usage(row, key):
    raw = row.get('raw_usage') or {}
    return fnum(raw.get(key), 0.0)

def response_issue(row):
    error = str(row.get('error') or '')
    response = str(row.get('response_text') or '')
    eval_result = row.get('eval_result') or {}
    stderr = str(eval_result.get('stderr') or '')
    errors = ' '.join(str(item) for item in (eval_result.get('errors') or []))
    leading_zero = 'leading zeros in decimal integer literals' in (stderr + errors)
    return bool(error) or not response.strip() or leading_zero

def summarize(group_rows, keys):
    groups = defaultdict(list)
    for row in group_rows:
        groups[tuple(row[key] for key in keys)].append(row)
    records = []
    for key_values, items in sorted(groups.items()):
        passed = sum(bool(row.get('passed')) for row in items)
        total = len(items)
        durations = [fnum(row.get('duration_seconds')) for row in items]
        record = {key: value for key, value in zip(keys, key_values)}
        record.update({
            'n': total,
            'passed': passed,
            'failed': total - passed,
            'pass_rate': passed / total if total else 0.0,
            'median_duration_seconds': statistics.median(durations) if durations else 0.0,
            'mean_duration_seconds': statistics.mean(durations) if durations else 0.0,
            'mean_prompt_tokens': statistics.mean([usage(row, 'prompt_tokens') for row in items]) if items else 0.0,
            'mean_completion_tokens': statistics.mean([usage(row, 'completion_tokens') for row in items]) if items else 0.0,
            'response_issue_rate': sum(response_issue(row) for row in items) / total if total else 0.0,
        })
        records.append(record)
    return records

def table(records, columns, formatters=None, max_rows=None):
    formatters = formatters or {}
    records = records[:max_rows] if max_rows else records
    parts = ['<table><thead><tr>']
    for col in columns:
        parts.append(f'<th>{html.escape(col)}</th>')
    parts.append('</tr></thead><tbody>')
    for record in records:
        parts.append('<tr>')
        for col in columns:
            value = record.get(col, '')
            if col in formatters:
                value = formatters[col](value)
            elif isinstance(value, float):
                value = f'{value:.3f}'
            parts.append(f'<td>{html.escape(str(value))}</td>')
        parts.append('</tr>')
    parts.append('</tbody></table>')
    display(HTML(''.join(parts)))

def bar_chart(records, label_key, value_key, title='', max_rows=None, formatter=None):
    records = records[:max_rows] if max_rows else records
    if title:
        display(Markdown(f'**{title}**'))
    max_value = max([abs(fnum(r.get(value_key))) for r in records] or [1])
    parts = ['<div style="font-family: ui-monospace, SFMono-Regular, Menlo, monospace; max-width: 980px;">']
    for record in records:
        label = str(record.get(label_key, ''))
        value = fnum(record.get(value_key))
        width = 2 if max_value == 0 else max(2, abs(value) / max_value * 560)
        shown = formatter(value) if formatter else f'{value:.3f}'
        color = '#3b82f6' if value >= 0 else '#ef4444'
        parts.append('<div style="display:flex;align-items:center;gap:8px;margin:4px 0;">')
        parts.append(f'<div style="width:250px;text-align:right;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;">{html.escape(label)}</div>')
        parts.append(f'<div style="height:16px;width:{width:.1f}px;background:{color};"></div>')
        parts.append(f'<div>{html.escape(shown)}</div></div>')
    parts.append('</div>')
    display(HTML(''.join(parts)))

summary_by_kind = summarize(rows, ('kind',))
summary_by_model_kind = summarize(rows, ('model_slug', 'kind'))
summary_by_model = summarize(rows, ('model_slug',))
summary_by_category = summarize(rows, ('model_slug', 'kind', 'category_id'))

display(Markdown(f'Loaded **{len(rows)}** final API result rows from `{RESULTS_DIR}`.'))


## 1. Task Kind Is The First Split

The API results should not be interpreted as one pooled score. Math and Python/code have different difficulty profiles and reverse some model rankings.


In [ ]:
table(summary_by_kind, ['kind', 'n', 'passed', 'failed', 'pass_rate', 'median_duration_seconds', 'mean_prompt_tokens', 'mean_completion_tokens', 'response_issue_rate'], {'pass_rate': pct, 'response_issue_rate': pct})
bar_chart(sorted(summary_by_kind, key=lambda r: r['pass_rate'], reverse=True), 'kind', 'pass_rate', 'Pass rate by task kind', formatter=lambda v: f'{100*v:.1f}%')


## 2. Model Ranking Depends On Task Kind

DeepSeek is much stronger on math, while Gemini is slightly stronger on Python. The overall score is therefore dominated by the mix of task domains.


In [ ]:
rows_mk = sorted(summary_by_model_kind, key=lambda r: (r['kind'], -r['pass_rate']))
for r in rows_mk:
    r['label'] = f"{r['kind']} / {r['model_slug']}"
table(rows_mk, ['model_slug', 'kind', 'n', 'passed', 'failed', 'pass_rate', 'median_duration_seconds', 'mean_completion_tokens', 'response_issue_rate'], {'pass_rate': pct, 'response_issue_rate': pct})
bar_chart(rows_mk, 'label', 'pass_rate', 'Model ranking by task kind', formatter=lambda v: f'{100*v:.1f}%')


## 3. Overall Model Results

This table is useful for headline reporting, but it should be paired with the task-kind split above.


In [ ]:
rows_model = sorted(summary_by_model, key=lambda r: -r['pass_rate'])
table(rows_model, ['model_slug', 'n', 'passed', 'failed', 'pass_rate', 'mean_duration_seconds', 'mean_prompt_tokens', 'mean_completion_tokens', 'response_issue_rate'], {'pass_rate': pct, 'response_issue_rate': pct})
bar_chart(rows_model, 'model_slug', 'pass_rate', 'Overall pass rate', formatter=lambda v: f'{100*v:.1f}%')


## 4. Pairwise Disagreement

A row-level comparison shows whether one model dominates or whether models solve different examples. This is more informative than only comparing aggregate accuracy.


In [ ]:
models = sorted({row['model_slug'] for row in rows})
comparison = []
for kind in sorted({row['kind'] for row in rows}):
    keyed = defaultdict(dict)
    for row in rows:
        if row['kind'] == kind:
            keyed[row['row_id']][row['model_slug']] = bool(row.get('passed'))
    if len(models) >= 2:
        a, b = models[0], models[1]
        both = [value for value in keyed.values() if a in value and b in value]
        comparison.append({
            'kind': kind,
            'model_a': a,
            'model_b': b,
            'n': len(both),
            'both_pass': sum(v[a] and v[b] for v in both),
            'a_only_pass': sum(v[a] and not v[b] for v in both),
            'b_only_pass': sum((not v[a]) and v[b] for v in both),
            'both_fail': sum((not v[a]) and (not v[b]) for v in both),
        })
table(comparison, ['kind', 'model_a', 'model_b', 'n', 'both_pass', 'a_only_pass', 'b_only_pass', 'both_fail'])


## 5. Category Sensitivity

`category_id` is the prompt/template variant axis. The strongest and weakest categories by model and kind identify where prompt framing interacts with task type and model behavior.


In [ ]:
cat = list(summary_by_category)
for r in cat:
    r['label'] = f"{r['model_slug']} / {r['kind']} / c{r['category_id']}"
strongest = sorted(cat, key=lambda r: -r['pass_rate'])[:12]
weakest = sorted(cat, key=lambda r: r['pass_rate'])[:12]
table(strongest, ['model_slug', 'kind', 'category_id', 'n', 'passed', 'pass_rate'], {'pass_rate': pct})
bar_chart(strongest, 'label', 'pass_rate', 'Strongest category slices', formatter=lambda v: f'{100*v:.1f}%')
table(weakest, ['model_slug', 'kind', 'category_id', 'n', 'passed', 'pass_rate'], {'pass_rate': pct})
bar_chart(weakest, 'label', 'pass_rate', 'Weakest category slices', formatter=lambda v: f'{100*v:.1f}%')


## 6. Runtime And Token Behavior

Latency and completion length differ sharply by model and task. This matters for interpreting usability, cost, and why some models may fail by producing too much or too little output.


In [ ]:
runtime_rows = sorted(summary_by_model_kind, key=lambda r: -r['median_duration_seconds'])
for r in runtime_rows:
    r['label'] = f"{r['model_slug']} / {r['kind']}"
table(runtime_rows, ['model_slug', 'kind', 'median_duration_seconds', 'mean_duration_seconds', 'mean_prompt_tokens', 'mean_completion_tokens'])
bar_chart(runtime_rows, 'label', 'median_duration_seconds', 'Median duration seconds')
completion_rows = sorted(summary_by_model_kind, key=lambda r: -r['mean_completion_tokens'])
bar_chart(completion_rows, 'label', 'mean_completion_tokens', 'Mean completion tokens')


## 7. Response Cleanliness And Error Modes

After reruns, the final API results should have no transport errors, empty responses, or known parser artifacts. This section verifies that before using the scores in thesis claims.


In [ ]:
issue_rows = []
for record in summary_by_model_kind:
    issue_rows.append({
        'model_slug': record['model_slug'],
        'kind': record['kind'],
        'n': record['n'],
        'response_issue_rate': record['response_issue_rate'],
        'response_issues': round(record['response_issue_rate'] * record['n']),
    })
table(issue_rows, ['model_slug', 'kind', 'n', 'response_issues', 'response_issue_rate'], {'response_issue_rate': pct})
if all(r['response_issues'] == 0 for r in issue_rows):
    display(Markdown('✅ No unresolved response issues remain in `results_final`.'))
else:
    display(Markdown('⚠️ Some unresolved response issues remain.'))


## 8. Prompt Framing Significance

This section tests tone, role, and format contrasts within each model and task kind. Treat these as exploratory tests: the prompt matrix is balanced, but the safest interpretation is still conditional by model and task kind.

Two checks are reported:

- pooled two-proportion z-tests over prompt rows,
- paired base-task sign tests over the 50 base tasks.

Both families are Holm-corrected across all tone/role/format contrasts.


In [ ]:
import math
import csv


def find_thesis_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '02ScalingPhrase' / 'localmodel' / 'problems').exists():
            return candidate
    raise FileNotFoundError('Could not locate thesis root with localmodel prompt templates')


def load_prompt_metadata() -> dict[tuple[str, str], dict[str, str]]:
    thesis_root = find_thesis_root(Path.cwd())
    template_paths = {
        'python': thesis_root / '02ScalingPhrase' / 'localmodel' / 'problems' / 'prompt_matrix_prompt_template.09.csv',
        'math': thesis_root / '02ScalingPhrase' / 'localmodel' / 'problems' / 'prompt_matrix_prompt_template.09.math.csv',
    }
    metadata = {}
    for kind, path in template_paths.items():
        with path.open(encoding='utf-8', newline='') as f:
            for row in csv.DictReader(f):
                role = row.get('Role framing', '').strip() or 'unknown'
                metadata[(kind, str(row['ID']))] = {
                    'tone': row.get('Tone', '').strip() or 'unknown',
                    'role_framing': role,
                    'role_binary': 'role' if role.lower() != 'no role' else 'no role',
                    'format': row.get('Format', '').strip() or 'unknown',
                }
    return metadata

prompt_metadata = load_prompt_metadata()
framing_rows = []
for row in rows:
    enriched = dict(row)
    enriched.update(prompt_metadata.get((row['kind'], str(row.get('category_id'))), {}))
    framing_rows.append(enriched)


def two_proportion_p_value(success_a, total_a, success_b, total_b):
    if total_a == 0 or total_b == 0:
        return 1.0
    pooled = (success_a + success_b) / (total_a + total_b)
    variance = pooled * (1 - pooled) * (1 / total_a + 1 / total_b)
    if variance <= 0:
        return 1.0
    z_score = abs((success_b / total_b) - (success_a / total_a)) / math.sqrt(variance)
    return math.erfc(z_score / math.sqrt(2))


def sign_test_p_value(positive, negative):
    n = positive + negative
    if n == 0:
        return 1.0
    k = min(positive, negative)
    lower_tail = sum(math.comb(n, i) for i in range(k + 1)) / (2 ** n)
    return min(1.0, 2 * lower_tail)


def holm_adjust(records, p_key, out_key):
    ordered = sorted(enumerate(records), key=lambda item: item[1][p_key])
    m = len(ordered)
    adjusted = [1.0] * m
    running = 0.0
    for rank, (original_index, record) in enumerate(ordered, start=1):
        value = min(1.0, (m - rank + 1) * record[p_key])
        running = max(running, value)
        adjusted[original_index] = running
    for record, value in zip(records, adjusted):
        record[out_key] = value


def pass_rate(items):
    return sum(bool(row.get('passed')) for row in items) / len(items) if items else 0.0

# Raw prompt-factor summary for context.
framing_summary = []
for factor in ['tone', 'role_binary', 'role_framing', 'format']:
    groups = defaultdict(list)
    for row in framing_rows:
        groups[(row['model_slug'], row['kind'], factor, row.get(factor, 'unknown'))].append(row)
    for (model, kind, _, level), items in sorted(groups.items()):
        framing_summary.append({
            'model_slug': model,
            'kind': kind,
            'factor': factor,
            'level': level,
            'n': len(items),
            'passed': sum(bool(row.get('passed')) for row in items),
            'pass_rate': pass_rate(items),
        })

with (ANALYSIS_DIR / 'api_prompt_factor_summary.csv').open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(framing_summary[0].keys()))
    writer.writeheader()
    writer.writerows(framing_summary)

display(Markdown('### Raw factor pass rates'))
table(
    [row for row in framing_summary if row['factor'] in {'tone', 'role_binary', 'format'}],
    ['model_slug', 'kind', 'factor', 'level', 'n', 'passed', 'pass_rate'],
    {'pass_rate': pct},
)

# Significance tests.
significance = []
for factor in ['tone', 'role_binary', 'format']:
    for model in sorted({row['model_slug'] for row in framing_rows}):
        for kind in sorted({row['kind'] for row in framing_rows}):
            subset = [row for row in framing_rows if row['model_slug'] == model and row['kind'] == kind]
            levels = sorted({row.get(factor, 'unknown') for row in subset})
            by_level = {level: [row for row in subset if row.get(factor, 'unknown') == level] for level in levels}
            by_base_level = defaultdict(lambda: defaultdict(list))
            for row in subset:
                by_base_level[row.get('task_id')][row.get(factor, 'unknown')].append(row)
            for i, level_a in enumerate(levels):
                for level_b in levels[i + 1:]:
                    rows_a = by_level[level_a]
                    rows_b = by_level[level_b]
                    passed_a = sum(bool(row.get('passed')) for row in rows_a)
                    passed_b = sum(bool(row.get('passed')) for row in rows_b)
                    paired_deltas = []
                    for _, level_rows in by_base_level.items():
                        if level_a in level_rows and level_b in level_rows:
                            paired_deltas.append(pass_rate(level_rows[level_b]) - pass_rate(level_rows[level_a]))
                    positive = sum(delta > 0 for delta in paired_deltas)
                    negative = sum(delta < 0 for delta in paired_deltas)
                    significance.append({
                        'model_slug': model,
                        'kind': kind,
                        'factor': factor,
                        'level_a': level_a,
                        'level_b': level_b,
                        'n_a': len(rows_a),
                        'passed_a': passed_a,
                        'pass_rate_a': pass_rate(rows_a),
                        'n_b': len(rows_b),
                        'passed_b': passed_b,
                        'pass_rate_b': pass_rate(rows_b),
                        'delta_b_minus_a_pp': 100 * (pass_rate(rows_b) - pass_rate(rows_a)),
                        'pooled_p': two_proportion_p_value(passed_a, len(rows_a), passed_b, len(rows_b)),
                        'paired_bases': len(paired_deltas),
                        'paired_mean_delta_pp': 100 * (statistics.mean(paired_deltas) if paired_deltas else 0.0),
                        'paired_positive_bases': positive,
                        'paired_negative_bases': negative,
                        'paired_tied_bases': sum(delta == 0 for delta in paired_deltas),
                        'paired_sign_p': sign_test_p_value(positive, negative),
                    })

holm_adjust(significance, 'pooled_p', 'pooled_holm_p')
holm_adjust(significance, 'paired_sign_p', 'paired_sign_holm_p')
for row in significance:
    row['pooled_significant_0_05'] = row['pooled_holm_p'] < 0.05
    row['paired_significant_0_05'] = row['paired_sign_holm_p'] < 0.05

sig_path = ANALYSIS_DIR / 'api_prompt_factor_significance.csv'
with sig_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(significance[0].keys()))
    writer.writeheader()
    writer.writerows(significance)

ranked = sorted(significance, key=lambda row: min(row['pooled_holm_p'], row['paired_sign_holm_p']))
display(Markdown('### Smallest adjusted p-values'))
table(
    ranked[:16],
    ['model_slug', 'kind', 'factor', 'level_a', 'level_b', 'delta_b_minus_a_pp', 'pooled_p', 'pooled_holm_p', 'paired_sign_p', 'paired_sign_holm_p'],
)

pooled_sig = [row for row in significance if row['pooled_significant_0_05']]
paired_sig = [row for row in significance if row['paired_significant_0_05']]
if not pooled_sig and not paired_sig:
    display(Markdown('**Result:** no tone, role, or format contrast is significant after Holm correction at α = 0.05. The largest observed effects should be reported as directional/exploratory, not confirmatory.'))
else:
    display(Markdown(f'**Result:** {len(pooled_sig)} pooled contrasts and {len(paired_sig)} paired sign-test contrasts are significant after Holm correction at α = 0.05.'))

display(Markdown(f'Full significance table written to `{sig_path.name}`.'))


## 8. Thesis-Level Interpretation Guardrails

- Do not report a single pooled API score without the math/code split.
- Treat DeepSeek and Gemini as complementary: DeepSeek dominates math, Gemini slightly leads Python/code.
- Pairwise disagreement matters: many rows are solved by only one model.
- Separate model ability from response cleanliness; final API outputs are now clean after reruns.
- Category/prompt effects should be interpreted conditionally by model and task kind, not as universal prompt rules.
